# Parametric Univariate Model Generate Samples
This notebook can be used to generate adjusted images using
the final architecture of the parametric generator.
Images can be read from a source directory and adjusted/edited images are saved into the target dir.

In [2]:
import torch
from torchvision import transforms
from social_media_overlay.models.regressor import EmotionRegressor
from social_media_overlay.models.parametric import (
    ConditionalParametricResidualMobileNetGenerator,
)
import pandas as pd


import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

sns.set_theme(style="white")

plt.rcParams.update({
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
})

%config InlineBackend.figure_format = 'svg'

FIG_SIZE = (7, 2.5)


In [3]:
device = torch.device('cuda')

# This block loads required dependencies
# We then load the cuda environment

print("Device Info")
print("Selected device:", device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU count:", torch.cuda.device_count())
    print("Current GPU index:", torch.cuda.current_device())
    print("GPU name:", torch.cuda.get_device_name(torch.cuda.current_device()))
    print("GPU capability:", torch.cuda.get_device_capability())
else:
    print("Running on CPU")

Device Info
Selected device: cuda
CUDA available: True
CUDA version: 12.6
GPU count: 8
Current GPU index: 0
GPU name: Tesla P100-SXM2-16GB
GPU capability: (6, 0)


### Configuration

In [ ]:
# checkpoint
# change this
RESUME_CHECKPOINT_PATH = "../data/models/chkp_param_raw_ref_arous_il0_clip5e-2_FILM_uni_480_b64_fc64-3_headfc64-2-sig_lr1e-4_1-1/chkp_param_raw_ref_arous_il0_clip5e-2_FILM_uni_480_b64_fc64-3_headfc64-2-sig_lr1e-4_1-1-final.pt"

IMAGE_RESIZE_SIZE = 480
IMAGE_CROP_SIZE = 480

BATCH_SIZE = 8
NUM_WORKERS = 8
PIN_MEMORY = True
DROP_LAST_BATCH = False

# regressor
REGRESSOR_MODEL_PATH = "../data/gebhardt/models/va_pred_all"
REGRESSOR_USE_AVERAGE = True
REGRESSOR_NORMALIZE_INPUT = True
REGRESSOR_USE_AROUSAL = True
FREEZE_REGRESSOR = True

# generator
GENERATOR_FC_DIM = 64
GENERATOR_FC_DEPTH = 3
GENERATOR_USE_PRETRAINED_BACKBONE = True
GENERATOR_FREEZE_BACKBONE = True

COLOR_HEAD_DEPTH = 2
COLOR_HEAD_DIM = 64
TONE_HEAD_DEPTH = 2
TONE_HEAD_DIM = 64

USE_FILM_CONDITIONING = True
USE_RES_HEAD = False
USE_ALPHA_HEAD = False
USE_SIDMOID_HEAD = True

data_transforms = transforms.Compose(
    [
        transforms.Resize(IMAGE_RESIZE_SIZE),
        transforms.CenterCrop(IMAGE_CROP_SIZE),
        transforms.ToTensor(),
    ]
)
regressor = EmotionRegressor(
    path_to_model=REGRESSOR_MODEL_PATH,
    average=REGRESSOR_USE_AVERAGE,
    device=device,
    normalize=REGRESSOR_NORMALIZE_INPUT,
).to(device)
if FREEZE_REGRESSOR:
    for param in regressor.model.parameters():
        param.requires_grad = False

regressor.eval()

### Load the model

In [6]:
model = ConditionalParametricResidualMobileNetGenerator(
    fc_dim=GENERATOR_FC_DIM,
    fc_depth=GENERATOR_FC_DEPTH,
    mobilenet_pretrained=GENERATOR_USE_PRETRAINED_BACKBONE,
    freeze_backbone=GENERATOR_FREEZE_BACKBONE,
    color_head_depth=COLOR_HEAD_DEPTH,
    color_head_dim=COLOR_HEAD_DIM,
    tone_head_depth=TONE_HEAD_DEPTH,
    tone_head_dim=TONE_HEAD_DIM,
    use_film=USE_FILM_CONDITIONING,
    use_res_head=USE_RES_HEAD,
    use_alpha_head=USE_ALPHA_HEAD,
    use_sigmoid_head=USE_SIDMOID_HEAD,
).to(device)
checkpoint = torch.load(RESUME_CHECKPOINT_PATH)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

In [8]:
from pathlib import Path
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import matplotlib.pyplot as plt

# CONFIG
image_directory = Path("../data/auswertung")
output_directory = Path("../data/auswertung/test2")
output_directory.mkdir(parents=True, exist_ok=True)

alpha_values = [0.1 ,0.125, 0.15 , 0.175,]

image_size = 256
batch_size = 1
num_workers = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

plot_cfg = {
    "fig_width_per_col": 2.5,
    "fig_height": 3.2,
    "title_fontsize": 9,
    "title_pad": 4,
    "wspace": 0.03,
    "top": 0.88,
    "bottom": 0.02,
    "left": 0.01,
    "right": 0.99,
    "save_format": "svg",
    "dpi": 300,
}

# CUSTOM DATASET / DATALOADER
class ImageFolderDataset(Dataset):
    def __init__(self, directory, transform=None):
        self.directory = Path(directory)
        self.transform = transform

        self.image_paths = sorted([
            p for p in self.directory.iterdir()
            if p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
        ])

        if len(self.image_paths) == 0:
            raise ValueError(f"No images found in {self.directory}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]

        img = Image.open(path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        return {
            "image": img,
            "path": str(path),
            "name": path.stem,
        }

dataset = ImageFolderDataset(
    directory=image_directory,
    transform=data_transforms,
)

data_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
)

# HELPERS
def _to_plot(img):
    img = img.detach().cpu().clamp(0, 1)

    if img.shape[0] == 1:
        return img.squeeze(0).numpy(), "gray"

    return img.permute(1, 2, 0).numpy(), None


def _set_safe_title(ax, text, fontsize=8, pad=3):
    title = ax.set_title(text, fontsize=fontsize, pad=pad)
    title.set_clip_on(False)
    return title


def _predict_valence_arousal(regressor, images):
    preds = regressor.predict(images)

    valence = preds[:, 0]
    arousal = preds[:, 1]

    return valence, arousal

# PLOTTING FUNCTION
def plot_alpha_manipulations(
    img,
    image_name,
    model,
    regressor,
    alpha_values,
    output_directory,
    plot_cfg,
    device,
):
    model.eval()

    img = img.to(device)

    num_cols = len(alpha_values) + 1

    fig, axes = plt.subplots(
        1,
        num_cols,
        figsize=(
            plot_cfg["fig_width_per_col"] * num_cols,
            plot_cfg["fig_height"],
        ),
    )

    if num_cols == 1:
        axes = [axes]

    with torch.no_grad():
        # Original image
        v0, a0 = _predict_valence_arousal(regressor, img)

        plot_img, cmap = _to_plot(img[0])

        axes[0].imshow(plot_img, cmap=cmap)
        _set_safe_title(
            axes[0],
            f"Original\nV={v0[0].item():.2f}, A={a0[0].item():.2f}",
            fontsize=plot_cfg["title_fontsize"],
            pad=plot_cfg["title_pad"],
        )
        axes[0].axis("off")

        # Manipulated images
        img_rep = img.repeat(len(alpha_values), 1, 1, 1)

        alphas = torch.tensor(
            alpha_values,
            dtype=torch.float32,
            device=device,
        )

        gen_imgs, _ = model(img_rep, alphas)

        valence, arousal = _predict_valence_arousal(regressor, gen_imgs)

        for j, alpha in enumerate(alpha_values):
            ax = axes[j + 1]

            plot_img, cmap = _to_plot(gen_imgs[j])

            ax.imshow(plot_img, cmap=cmap)
            _set_safe_title(
                ax,
                f"alpha={alpha}\nV={valence[j].item():.2f}, A={arousal[j].item():.2f}",
                fontsize=plot_cfg["title_fontsize"],
                pad=plot_cfg["title_pad"],
            )
            ax.axis("off")

    plt.subplots_adjust(
        wspace=plot_cfg["wspace"],
        top=plot_cfg["top"],
        bottom=plot_cfg["bottom"],
        left=plot_cfg["left"],
        right=plot_cfg["right"],
    )

    save_path = output_directory / f"{image_name}_alpha_plot.{plot_cfg['save_format']}"

    plt.savefig(
        save_path,
        bbox_inches="tight",
        pad_inches=0.02,
        dpi=plot_cfg["dpi"],
    )

    plt.close(fig)

    return save_path


# RUN
model = model.to(device)
model.eval()

saved_paths = []

for batch in data_loader:
    images = batch["image"]
    image_names = batch["name"]

    for i in range(images.shape[0]):
        img = images[i:i + 1]
        image_name = image_names[i]

        save_path = plot_alpha_manipulations(
            img=img,
            image_name=image_name,
            model=model,
            regressor=regressor,
            alpha_values=alpha_values,
            output_directory=output_directory,
            plot_cfg=plot_cfg,
            device=device,
        )

        saved_paths.append(save_path)

print(f"Saved {len(saved_paths)} plots to {output_directory}")

Saved 9 plots to ../data/auswertung/test2
